In [1]:
import sys
sys.path.append('../../..') #This line makes it possible to access scripts and settings, which are above notebooks in heirarchy
#the above line also makes it possible to access files from the root with out a ROOT_DIR
import subprocess
import importlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from numpy.lib.recfunctions import drop_fields, structured_to_unstructured

from notebooks.preprocessing.phase2preprocess import preprocess_general, load_parquet, prpsettings
# from scripts.generate_dataset import generate_dataset
from scripts.dataImport import impsettings

from notebooks.preprocessing.phase2to3connection import filter_drop

In [2]:
BASE_DIR = Path("__file__").resolve().parent

In [3]:
PHASES = {
    "phase2": "phase_2",
    "phase3": "phase_3"
}

In [4]:
def import_phases(name):
    nb = BASE_DIR / f"{name}.ipynb"

    if nb.exists():
        subprocess.run(
            [sys.executable, "-m", "nbconvert", "--to", "script", "--output", name, str(nb)],
            cwd=BASE_DIR,
            check=True,
            capture_output=True,
            text=True,
        )

    if str(BASE_DIR) not in sys.path:
        sys.path.insert(0, str(BASE_DIR))

    importlib.invalidate_caches()
    return importlib.import_module(name) if name not in sys.modules else importlib.reload(sys.modules[name])

In [5]:
def get_phases(phases=None):
    if phases is None:
        names = list(PHASES)
    if isinstance(phases, str):
        phases = [phases]
    return {phase: import_phases(PHASES[phase]) for phase in phases}

In [6]:
def get_packet_data():
    """Get training data for phase 2"""
    #create the dataset if it doesn't already exist
    input_data = impsettings.PROCESSED_DATA_PATH / "flow_training_sample.parquet"
    if not input_data.is_file():
        print('Constructing Datasets...')
        %run ../../../scripts/generate_dataset.py
    #import wwt from data/processed
    raw = load_parquet(impsettings.PROCESSED_DATA_PATH, 'combined_sample.parquet')
    #seperate out labels
    y = raw['pkt_label']
    raw = drop_fields(raw, 'pkt_label', asrecarray=True)
    #Seperate out temp and test data
    X_temp, X_test, y_temp, y_test = train_test_split(raw, y, test_size=0.15, stratify=y, random_state=42)

    #preprocess training data first
    print('preprocess training/validation data')
    X_temp_pp = preprocess_general(array=X_temp, split='Train')

    #seperate out packet features from wwt for training/testing
    # pkt_features = [feat for feat in X_temp_pp.dtype.names if feat.startswith('pkt_')]
    # X_temp_pp_flow = X_temp_pp[pkt_features]

    #preprocess testing data, using mean, std, and onehot labels learned from training
    print('preprocess testing data')
    X_test_pp = preprocess_general(array=X_test, split='Test')
    X_train, X_val, y_train, y_val = train_test_split(X_temp_pp, y_temp, test_size=0.15, stratify=y_temp, random_state=42)

    #change to dataframes for compatability with model train functions.
    return (
        pd.DataFrame(X_train), 
        pd.DataFrame(X_val), 
        pd.DataFrame(X_test_pp), 
        pd.Series(y_train), 
        pd.Series(y_val), 
        pd.Series(y_test)
    )

In [7]:
def get_flow_data():
    """Get training data for phase 3"""
    #check that phase 2 data has been preprocessed first. Essential to learn preprocessing states.
    input_data = impsettings.PROCESSED_DATA_PATH / "flow_training_sample.parquet"
    prpsettings.load_state()
    if not input_data.is_file():
        raise RuntimeError('Must run generate_dataset() through phase_2 get_training_data() first.')
    if prpsettings.encoder is None or prpsettings.mean is None or prpsettings.std is None:
        raise RuntimeError('Must learn preprocessing states on wwt through phase_2 get_training_data() first.')

    #import phase 3 training data from data/processed
    raw = load_parquet(impsettings.PROCESSED_DATA_PATH, 'flow_training_sample.parquet')

    #seperate out labels
    y = raw['label']
    raw = drop_fields(raw, 'label', asrecarray=True)
    #preprocess training data
    print('preprocessing training data')
    raw_pp = preprocess_general(array=raw, split='Flow')
    #no need to seperate training and testing data. The whole of flow_training_sample is used for training.
    X_train, X_val, y_train, y_val = train_test_split(
        raw_pp, y, test_size=0.15, stratify=y, random_state=42
    )

    return (
        pd.DataFrame(X_train), 
        pd.DataFrame(X_val),
        pd.Series(y_train),
        pd.Series(y_val)
        )

In [8]:
def filter_wwt(wwt: pd.DataFrame, split:str):
    """Filter the WWT to packet or flow data. Split must be either 'Flow' or 'Packet'."""
    if split.lower() == 'packet':
        pkt_features = [feat for feat in wwt.columns if feat.startswith('pkt_')]
        return wwt[pkt_features]
    if split.lower() == 'flow':
        flow_features = [feat for feat in wwt.columns if feat.startswith('flow_')]
        return wwt[flow_features]
    else:
        raise ValueError("Split must be one of 'Flow' or 'Packet'")

# IMPORT PHASES

In [9]:
phases = get_phases(["phase2", "phase3"])

In [10]:
phase2 = phases["phase2"]
phase3 = phases["phase3"]

# PREPARE DATA

### PACKET DATA

In [11]:
X_train, X_val, X_test, y_train, y_val, y_test = get_packet_data()

preprocess training/validation data


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


preprocess testing data


In [12]:
print('\nFilter flow features out of training set for anomoly detectors.')
X_train_pkt = filter_wwt(X_train, split='Packet')
X_val_pkt = filter_wwt(X_val, split='Packet')
X_test_pkt = filter_wwt(X_test, split='Packet')

print('Filter flow features out of testing for unsupervised anomoly detection.')
X_test_pkt = filter_wwt(X_test, split='Packet')


Filter flow features out of training set for anomoly detectors.
Filter flow features out of testing for unsupervised anomoly detection.


### FLOW DATA

In [13]:
X_train_flow, X_val_flow, y_train_flow, y_val_flow = get_flow_data()

preprocessing training data


# TRAIN PHASES

### PHASE 2  >>>>>

In [14]:
trained_phase2_models = phase2.train_all(X_train_pkt, X_val_pkt, y_train, y_val)

ISOLATION FOREST
Isolation Forest fitted.
Threshold tuned: 0.03019  (val F1=0.0497)
              precision    recall  f1-score   support

      Benign       0.98      0.98      0.98     25500
      Attack       0.06      0.04      0.05       535

    accuracy                           0.96     26035
   macro avg       0.52      0.51      0.52     26035
weighted avg       0.96      0.96      0.96     26035

Isolation forest done training...
Saved > /Users/pauligbinedion/MASTERS/SUMMER/CAPSTONE/ECE597-Capstone-IoT-IDS/notebooks/saved_models/isolation_forest.joblib
AUTOENCODER
Threshold tuned: 0.079771  (val F1=0.6152)
              precision    recall  f1-score   support

      Benign       0.99      1.00      0.99     25500
      Attack       0.78      0.51      0.62       535

    accuracy                           0.99     26035
   macro avg       0.89      0.75      0.80     26035
weighted avg       0.99      0.99      0.99     26035

Autoencoder done training...
Saved > /Users/paul

### PHASE 3 >>>>>

In [15]:
phase3_mlp = phase3.train_model("mlp", X_train_flow, X_val_flow, y_train_flow, y_val_flow)
phase3_rf = phase3.train_model("random_forest", X_train_flow, X_val_flow, y_train_flow, y_val_flow)

#train XGBoost from phase_3 directly because of a threading error to be resolved soon
#phase3_xgb = phase3.train_model("xgboost", X_train_flow, X_val_flow, y_train_flow, y_val_flow)


MLP
Training on 173,566 samples across 6 classes.
Classes: ['Benign', 'Brute Force', 'DDoS-HTTP Flood', 'DNS Spoofing', 'DoS-HTTP Flood', 'XSS']
Features scaled with StandardScaler.
MLP fitted (26 iterations).

Accuracy: 0.9926 | F1-macro: 0.5730
                 precision    recall  f1-score   support

         Benign       0.99      1.00      1.00     30001
    Brute Force       0.77      0.26      0.39       140
DDoS-HTTP Flood       0.96      0.89      0.92       225
   DNS Spoofing       1.00      0.19      0.31        27
 DoS-HTTP Flood       0.94      0.72      0.82       229
            XSS       0.00      0.00      0.00         8

       accuracy                           0.99     30630
      macro avg       0.78      0.51      0.57     30630
   weighted avg       0.99      0.99      0.99     30630

MLP done training...
Saved > /Users/pauligbinedion/MASTERS/SUMMER/CAPSTONE/ECE597-Capstone-IoT-IDS/notebooks/saved_models/mlp.joblib

RANDOM FOREST
Training on 173,566 samples acr

# RUN PHASES PIPELINE

### PHASE 2 >>>>>

In [16]:
print('\nAnomoly detection on testing data')
X_arr_pkt, score_arr_pkt, label_arr_pkt = phase2.predict_phase2(X_test_pkt)


Anomoly detection on testing data


#### PROCESS RESPONSE FOR PHASE 3

In [17]:
print('\nAsserting that X_test_pkt and X_arr_pkt are the same to confirm that predict_phase2 has not changed order')
np.testing.assert_allclose(X_test_pkt.values, X_arr_pkt)


Asserting that X_test_pkt and X_arr_pkt are the same to confirm that predict_phase2 has not changed order


In [18]:
print(f"\nLabel the full testing set with anomoly detection labels. \nX_test shape: {X_test.shape}. label_arr_pkt shape: {label_arr_pkt.shape}")
X_test['phase_2_label'] = label_arr_pkt.ravel()


Label the full testing set with anomoly detection labels. 
X_test shape: (30630, 863). label_arr_pkt shape: (1, 30630)


##### REMOVE BENIGN PREDICTIONS

In [19]:
print('\nFilter out non-anomolous rows, and remove the anomoly detection labels')
filtered_X = filter_drop(df=X_test, label_col_idx=-1)


Filter out non-anomolous rows, and remove the anomoly detection labels


In [20]:
print('\nAsserting that the filtered_X has the same columns as X_test (minus the new label column in X_test)')
assert filtered_X.shape[1] == len(X_test.columns[:-1])
labeled_wwt = pd.DataFrame(filtered_X, columns=X_test.columns[:-1])


Asserting that the filtered_X has the same columns as X_test (minus the new label column in X_test)


##### GET FLOW DATA FOR PHASE 3

In [21]:
#filter out packet features from wwt
print('\nFilter packet features out of testing for supervised calssification.')
X_test_flow = filter_wwt(labeled_wwt, split='Flow')


Filter packet features out of testing for supervised calssification.


In [25]:
X_test_flow

,flow_Protocol,flow_Flow Duration,flow_Total Fwd Packet,flow_Total Bwd packets,flow_Total Length of Fwd Packet,flow_Total Length of Bwd Packet,flow_Fwd Packet Length Max,flow_Fwd Packet Length Min,flow_Fwd Packet Length Mean,flow_Fwd Packet Length Std,...,flow_Fwd Act Data Pkts,flow_Fwd Seg Size Min,flow_Active Mean,flow_Active Std,flow_Active Max,flow_Active Min,flow_Idle Mean,flow_Idle Std,flow_Idle Max,flow_Idle Min
0,-0.506668,-0.577896,-0.455873,-0.442168,-0.384719,-0.280586,1.075431,-0.309616,1.349835,1.764103,...,-0.452211,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
1,-0.506668,-0.578041,-0.458537,-0.444537,-0.415549,-0.280586,0.607766,-0.309616,1.320853,1.520835,...,-0.45489,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
2,-0.506668,-0.578665,-0.458137,-0.447192,-0.459445,1.858902,0.022549,-0.309616,1.389573,0.410642,...,-0.460364,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
3,-0.506668,-0.578675,-0.465842,-0.452195,-0.506553,0.171537,-0.967679,-0.309616,-0.911183,-0.862163,...,-0.463054,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
4,-0.506668,-0.578604,-0.464958,-0.452099,-0.506567,0.181977,-0.961377,-0.309616,-0.919485,-0.907572,...,-0.463054,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,-0.506668,-0.577535,-0.451894,-0.437197,-0.337448,-0.280586,1.426391,-0.309616,1.390414,1.657602,...,-0.448223,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
522,-0.506668,-0.578668,-0.462916,-0.449741,-0.50655,1.331166,-0.95762,-0.309616,-0.918305,-0.949002,...,-0.463065,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
523,-0.506668,-0.578694,-0.466811,-0.453289,-0.506622,-0.287444,-1.030333,-0.309616,-0.925519,-0.983936,...,-0.463096,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752
524,-0.506668,-0.578583,-0.464779,-0.451844,-0.506533,0.278803,-0.961377,-0.309616,-0.916616,-0.88482,...,-0.463044,0.70422,-0.337739,-0.282996,-0.491784,-0.133452,-0.418225,-0.20792,-0.59098,-0.226752


### PHASE 3 >>>>>>

In [27]:
#final_prediction_mlp = phase3.predict_phase3(X_test_flow, "mlp")
final_prediction_rf = phase3.predict_phase3(X_test_flow, "random_forest")

ValueError: X has 77 features, but RandomForestClassifier is expecting 78 features as input.